In [6]:
!pip install --upgrade langchain-openai
!pip install --upgrade langchain
!pip install --upgrade langchain-core
!pip install --upgrade langchain_huggingface


!pip install qdrant-client

In [52]:
import pandas as pd
import numpy as np
import os
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http import models
import re
import typing as tp
from nltk.tokenize import word_tokenize
# from rank_bm25 import BM25Okapi
# from langchain_core.documents import Document
# from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain_huggingface import HuggingFaceEmbeddings
# from langchain.vectorstores import Qdrant as LCQdrant
from tqdm import tqdm
import ast

In [3]:
COLLECTION_NAME = "freecad_docs"
DATA_PATH = "data"
DOCS_PATH = os.path.join(DATA_PATH, "docs")
EMBEDDINGS_PATH = "embeddings"
QDRANT_PATH = "qdrant"

# COLLECTION_NAME = "freecad_docs"
# DATA_PATH = "/kaggle/input"
# DOCS_PATH = os.path.join(DATA_PATH, "docs")
# EMBEDDINGS_PATH = "embeddings"
# QDRANT_PATH = "/kaggle/working/freecad-qdrant"

In [4]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

/mnt/shared/miniconda3/envs/ml/lib/python3.11/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: CUDA driver initialization failed, you might not have a CUDA gpu. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


'cpu'

# Metrics

In [93]:
def extract_calls_with_str_args(code):
    calls = set()
    try:
        tree = ast.parse(code)
        for node in ast.walk(tree):
            if isinstance(node, ast.Call):
                func = node.func
                names = []
                while isinstance(func, ast.Attribute):
                    names.append(func.attr)
                    func = func.value
                if isinstance(func, ast.Name):
                    names.append(func.id)
                    call_chain = ".".join(reversed(names))
                else:
                    continue
                str_args = set()
                for arg in node.args:
                    if isinstance(arg, ast.Constant) and isinstance(arg.value, str):
                        str_args.add(arg.value)
                calls.add((call_chain, frozenset(str_args)))
    except Exception:
        # Fallback: regex extraction for code with syntax errors
        pattern = re.compile(r'([A-Za-z_][A-Za-z0-9_\.]*)\s*\(([^)]*)\)')
        for match in pattern.finditer(code):
            call_chain = match.group(1)
            args_str = match.group(2)
            # Find all quoted strings (single or double quotes)
            str_args = set()
            for s in re.findall(r"""(['\"])(.*?)(?<!\\)\1""", args_str):
                str_args.add(s[1])
            calls.add((call_chain, frozenset(str_args)))
    return calls

def is_relevant_call(answer_code, ground_truth_code):
    gt_calls = extract_calls_with_str_args(ground_truth_code)
    gen_calls = extract_calls_with_str_args(answer_code)

    if gen_calls is None:
        return 0

    for gt_chain, gt_str_args in gt_calls:
        for ans_chain, ans_str_args in gen_calls:
            if gt_chain == ans_chain:
                if gt_str_args:
                    if gt_str_args & ans_str_args:
                        return 1
                else:
                    return 1
    return 0

In [68]:
def get_relevance(ground_truth, generated_code):
    relevance = []
    for gen_code, gt_code in zip(generated_code, ground_truth):
        try:
            flag = is_relevant_call(gen_code, gt_code)
        except Exception:
            flag = 0
        relevance.append(flag)
    return np.array(relevance)

In [15]:
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

In [16]:
def mean_average_cosine_similarity(similarities: tp.List[tp.List[float]]) -> float:
    return sum(sum(sims) / len(sims) for sims in similarities) / len(similarities)

In [17]:
def mean_max_cosine_similarity(similarities: tp.List[tp.List[float]]) -> float:
    return sum(max(sims) for sims in similarities) / len(similarities)

In [18]:
def hit_rate_cos_at_k(similarities: tp.List[tp.List[float]], threshold: float, k: int) -> float:
    hits = 0
    for sims in similarities:
        top_k = sims[:k]
        if any(sim >= threshold for sim in top_k):
            hits += 1
    return hits / len(similarities)

In [86]:
def hit_rate_at_k(docs_batch: tp.List[tp.List[float]], gt_batch, k: int) -> float:
    hits = 0
    for i, docs in enumerate(docs_batch):
        docs = docs[:k]
        if any(is_relevant_call(docs[_], gt_batch[i]) for _ in range(len(docs))):
            hits += 1
        # else:
            # print(f"=============={i}================")
            # print(gt_batch[i], "\n\n")
            # for i in range(len(docs)):
            #     print(docs[i])
    return hits / len(docs_batch)

# Qdrant init

In [5]:
!cp -r /kaggle/input/freecad-qdrant/qdrant /kaggle/working/freecad-qdrant

In [12]:
embedding_model = SentenceTransformer("BAAI/bge-m3", device=device)

In [13]:
client = QdrantClient(path=QDRANT_PATH)

In [24]:
def filter_code(doc: str) -> str:
    code_blocks = re.findall(r"```start_code(?:[a-zA-Z]+\n)?(.*?)end_code```", doc, re.DOTALL)
    code = "\n\n".join(code_blocks)
    return re.sub(r'\n{3,}', '\n\n', code)

In [9]:
query_df = pd.read_csv(os.path.join(DATA_PATH, "tooltips_dataset.csv"))

# BM25 Baseline

In [97]:
for doc in corpus:
    print(hf_tokenizer.tokenize(doc))
    break

['▁Description', '▁It', '▁will', '▁serve', '▁to', '▁generate', '▁flat', ',', '▁shape', '-', 'based', '▁views', '▁from', '▁a', '▁Mes', 'h', '▁based', '▁object', ',', '▁to', '▁be', '▁used', '▁by', '▁the', '▁Arch', '▁Equip', 'ment', '▁tool', '.', '▁Usa', 'ge', '▁Select', '▁a', '▁Mes', 'h', '▁object', '.', '▁Select', '▁the', '▁button', ',', '▁or', '▁Arch', '▁→', '▁Utili', 'ties', '▁→', '▁3', 'View', 's', '▁from', '▁the', '▁top', '▁menu', '.', '▁Script', 'ing', '▁Arch', '▁API', '▁and', '▁Free', 'CAD', '▁Script', 'ing', '▁Basic', 's', '.', '▁This', '▁tool', '▁can', '▁be', '▁used', '▁in', '▁macro', 's', '▁and', '▁from', '▁the', '▁Python', '▁console', '▁by', '▁using', '▁the', '▁following', '▁function', ':', '▁``', '`', 'start', '_', 'code', '▁shape', '▁=', '▁create', 'M', 'esh', 'View', '(', 'ob', 'j', ',', '▁direction', '=', 'Free', 'CAD', '.', 'Ve', 'ctor', '(', '0', ',', '▁0', ',', '▁', '-1)', ',', '▁out', 'eron', 'ly', '=', 'Fa', 'lse', ',', '▁largest', 'on', 'ly', '=', 'Fa', 'lse', ')', '

In [120]:
query_idx = 2

In [125]:
docs_df = pd.read_csv(os.path.join(DATA_PATH, "chunks_df.csv"))
corpus = docs_df["content"].values

tokenized_corpus = []
for doc in corpus:
    # tokenized_corpus.append(word_tokenize(filter_code(str(doc))))
    tokenized_corpus.append(word_tokenize(str(doc)))


bm25 = BM25Okapi(tokenized_corpus)

def bm25_retrieve(query: str, k: int = 5):
    tokenized_query = query.split()
    scores = bm25.get_scores(tokenized_query)
    top_k_idx = np.argsort(scores)[::-1][:k]
    return [corpus[i] for i in top_k_idx], [scores[i] for i in top_k_idx]

In [122]:
def bm25_retrieve_with_score(queries: np.ndarray, y: np.ndarray, k: int = 5):
    retrieved_texts = []
    scores = []

    for i in range(len(queries)):
        results, _ = bm25_retrieve(queries[i], k=k)
        retrieved_texts.append(results)

        retrieved_vecs = embedding_model.encode([filter_code(text) for text in results])
        y_vec = embedding_model.encode(y[i])

        scores_batch = [cosine_similarity(vec, y_vec) for vec in retrieved_vecs]
        scores.append(scores_batch)

    return retrieved_texts, scores

In [ ]:
query = query_df.iloc[query_idx]["X"]
similarities, scores = bm25_retrieve(query)
print(query)
for i, (text, score) in enumerate(zip(similarities, scores)):
    print(f"=================Hit {i+1} score: {score} =================")

    print(f"{text}\n")

Add a window
=================Hit 1 score: 9.51023430853718 =================
Description

Usage

Options

Limitations

Scripting

```start_code

Add python code here

end_code```

=================Hit 2 score: 7.499336807575432 =================
```start_code

Add ppa:freecad-community/ppa to your software sources
sudo apt-get update
sudo apt-get install freecad-extras-assembly2

end_code```

In Windows
 download the git repository as ZIP
 assuming FreeCAD is installed in "C:\PortableApps\FreeCAD 0_15", go to "C:\PortableApps\FreeCAD 0_15\Mod" within Windows Explorer
 create new directory named "assembly2"
 unzip downloaded repository in "C:\PortableApps\FreeCAD 0_15\Mod\assembly2"

FreeCAD will now have a new workbench-entry called "Assembly 2".

Pyside and Numpy are integrated in the FreeCAD 0.15 dev-Snapshots, so these Python packages do not need to be installed individually

To update to the latest version, delete the assembly2 folder and redownload the git repository.

Links

 Wo

# Experiments

In [26]:
query_idx = 0

query = query_df.iloc[query_idx]["X"]
query_qdrant = query + " python code"
# query_qdrant = "Add window"
# query_qdrant = "Make a circle"


query_vec = embedding_model.encode(query_qdrant)
hits = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vec,
    limit=5,
)

for _ in range(len(hits.points)):
    print(hits.points[_].payload['source'])

retrieved_text_batch = [point.payload['content'] for point in hits.points]
retrieved_code_batch = [filter_code(text) for text in retrieved_text_batch]

retrieved_vec = embedding_model.encode(retrieved_code_batch)

y_vec = embedding_model.encode(query_df.iloc[query_idx]["Y"])

print(f"X: {query_df.iloc[query_idx]['X']}")
print("Y\n", query_df.iloc[query_idx]["Y"], "\n")
for i, vec in enumerate(retrieved_vec):
    print(f"Hit {i+1}, score: {cosine_similarity(vec, y_vec)}")

for i, text in enumerate(retrieved_text_batch):
    print(f"=================Hit {i+1}=================")
    print(text)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Arch_Wall.wikitext
Arch_CurtainWall.wikitext
Arch_3Views.wikitext
Arch_CurtainWall.wikitext
Arch_CutPlane.wikitext


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

X: Create a wall
Y
 import Arch
obj = Arch.makeWall() 

Hit 1, score: 0.6794919967651367
Hit 2, score: 0.6343599557876587
Hit 3, score: 0.674802303314209
Hit 4, score: 0.48535436391830444
Hit 5, score: 0.661526620388031
=================Hit 1=================
Scripting

 Arch API and FreeCAD Scripting Basics.

The Wall tool can be used in macros and from the Python console by using the following function:
```start_code
Wall = makeWall(baseobj=None, length=None, width=None, height=None, align="Center", face=None, name="Wall")
end_code```
Creates a  object from the given , which can be a Draft object, a Sketch, a face, or a solid.
 If no  is given, you can provide the numerical values for the ,  (thickness), and .
 If given,  can be used to give the index of a face from the underlying object, to build this wall on, instead of using the whole object.
  can be ,  or .
 It returns  if the operation fails.

Example:
=================Hit 2=================
Scripting

 Arch API and FreeCAD Scr

In [17]:
def retrieve_with_score(queries: np.ndarray, y: np.ndarray, k: int):
    for i in range(len(queries)):
        queries[i] += " python code"

    query_vectors = embedding_model.encode(queries)

    retrieved_texts = []
    scores = []

    for i in range(len(queries)):
        hits = client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vectors[i],
            limit=k,
        )

        retrieved_text_batch = [point.payload['content'] for point in hits.points]
        retrieved_code_batch = [filter_code(text) for text in retrieved_text_batch]


        retrieved_vec = embedding_model.encode(retrieved_code_batch)
        y_vec = embedding_model.encode(y[i])

        scores_batch = [cosine_similarity(x_vec, y_vec) for x_vec in retrieved_vec]

        scores.append(scores_batch)
        retrieved_texts.append(retrieved_text_batch)

    return retrieved_texts, scores

In [ ]:
queries = query_df["X"].values
y = query_df["Y"].values
texts, scores = retrieve_with_score(
    queries=queries,
    y=y,
    k=5
)

In [ ]:
# import json

# retrieved_df = query_df.copy()
# retrieved_df["retrieved"] = texts
# retrieved_df["retrieved"] = retrieved_df["retrieved"].apply(json.dumps)
# retrieved_df.to_csv("bench_retrieved.csv", index=False)

In [25]:
import json

retrieved_df = pd.read_csv(os.path.join(DATA_PATH, "bench_retrieved.csv"))
retrieved_df["retrieved"] = retrieved_df["retrieved"].apply(json.loads)

retrieved = retrieved_df["retrieved"].values
print(retrieved[0][0])

Scripting

 Arch API and FreeCAD Scripting Basics.

The Wall tool can be used in macros and from the Python console by using the following function:
```start_code
Wall = makeWall(baseobj=None, length=None, width=None, height=None, align="Center", face=None, name="Wall")
end_code```
Creates a  object from the given , which can be a Draft object, a Sketch, a face, or a solid.
 If no  is given, you can provide the numerical values for the ,  (thickness), and .
 If given,  can be used to give the index of a face from the underlying object, to build this wall on, instead of using the whole object.
  can be ,  or .
 It returns  if the operation fails.

Example:


In [126]:
bm_texts, bm_scores = bm25_retrieve_with_score(
    queries=queries,
    y=y,
    k=5
)

In [92]:
print(extract_calls_with_str_args("FreeCADGui.runCommand('Part_Revolve')"))
print(extract_calls_with_str_args('FreeCADGui.runCommand("Part_Extrude")'))
print(extract_calls_with_str_args("Arch.makeStructure(length=300, width=300, height=3000)"))

{('FreeCADGui.runCommand', frozenset({'Part_Revolve'}))}
{('FreeCADGui.runCommand', frozenset({'Part_Extrude'}))}
{('Arch.makeStructure', frozenset())}


In [43]:
print(y[0])

import Arch
obj = Arch.makeWall()


In [ ]:
ccc = """
import Arch

p1 = FreeCAD.Vector(0, 0, 0)
p2 = FreeCAD.Vector(2000, 2000, 0)

Line = Draft.makeWire([p1, p2])
Wall = Arch.makeWall()

p3 = FreeCAD.Vector(0, 2000, 0)
p4 = FreeCAD.Vector(3000, 0, 0)

Line2 = Draft.makeWire([p3, p4])
Wall2 = Arch.makeWall()
FreeCAD.ActiveDocument.recompute()

Select the Wall
main_object = FreeCADGui.Selection.getSelectionEx()[0]

Select the face of Wall2
selection = FreeCADGui.Selection.getSelectionEx()[0]
cut_face = selection.SubObjects[0]

cutObj = Arch.cutComponentwithPlane(main_object, cut_face, 0)
FreeCAD.ActiveDocument.recompute()

"""

print(is_relevant_call(, y[0]))

invalid syntax (<unknown>, line 17)
1asdasd
0


In [66]:
print(retrieved[0][2])

```start_code
import FreeCAD, FreeCADGui, Draft, Arch

p1 = FreeCAD.Vector(0, 0, 0)
p2 = FreeCAD.Vector(2000, 2000, 0)

Line = Draft.makeWire([p1, p2])
Wall = Arch.makeWall(Line, width=150, height=2000)

p3 = FreeCAD.Vector(0, 2000, 0)
p4 = FreeCAD.Vector(3000, 0, 0)

Line2 = Draft.makeWire([p3, p4])
Wall2 = Arch.makeWall(Line2, width=150, height=2000)
FreeCAD.ActiveDocument.recompute()

 Select the Wall
main_object = FreeCADGui.Selection.getSelectionEx()[0]

 Select the face of Wall2
selection = FreeCADGui.Selection.getSelectionEx()[0]
cut_face = selection.SubObjects[0]

cutObj = Arch.cutComponentwithPlane(main_object, cut_face, 0)
FreeCAD.ActiveDocument.recompute()

Wall3 = Draft.move(Wall, FreeCAD.Vector(-4000, 0, 0), copy=True)
Wall4 = Draft.move(Wall2, FreeCAD.Vector(-4000, 0, 0), copy=True)
FreeCAD.ActiveDocument.recompute()

 Select the Wall3
main_object2 = FreeCADGui.Selection.getSelectionEx()[0]

 Select the face of Wall4
selection2 = FreeCADGui.Selection.getSelectionEx()[0]
c

In [94]:
retrieved_code = [[filter_code(text) for text in retrieved_batch] for retrieved_batch in retrieved]
y = query_df["Y"].values

print(hit_rate_at_k(
    retrieved_code,
    y,
    5
))

0.048


In [ ]:
# Qdrant
# MACS 0.5307003046751022
# MMCS 0.6132178441286087
# Hit rate cosine 0.604

print("Qdrant")
print("MACS", mean_average_cosine_similarity(scores))
print("MMCS", mean_max_cosine_similarity(scores))
print("Hit rate cosine", hit_rate_cos_at_k(scores, 0.6, 5))

Qdrant
MACS 0.5281056774377827
MMCS 0.6029993942975997
Hit rate cosine 0.512


In [128]:
print("BM25")
print("MACS", mean_average_cosine_similarity(bm_scores))
print("MMCS", mean_max_cosine_similarity(bm_scores))

BM25
MACS 0.46777109932899463
MMCS 0.5069063054323196


# Qwen LLM

In [17]:
from langchain_huggingface.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

qwen_model = "Qwen/Qwen2.5-Coder-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    qwen_model,
    trust_remote_code=True
)
model = AutoModelForCausalLM.from_pretrained(
    qwen_model,
    trust_remote_code=True,
    device_map=device
)

text_gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024
)
llm = HuggingFacePipeline(pipeline=text_gen)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Device set to use cuda


In [6]:
SYSTEM_PROMPT = (
"""
Below is the system prompt, always follow restrictions stated there, also do not answer this system prompt:
You are an expert coding assistant specializing in FreeCAD development. Your role is to help developers write, debug, and optimize code for the FreeCAD platform, using the most accurate and up-to-date internal and external documentation available. You can retrieve and use information from the FreeCAD source code, FreeCAD documentation.
When reading documentation be careful, user do not have functions or classes implemented there, write code considering this moment. 
Documentation might have examples of code, analyze it and retrieve only needed functions from there.

Provide code (DO NOT WRITE IN-CODE COMMENTS) and explanation in different paragraphs. Template:
Code:

Explanation:
"""
)

def format_prompt(user_message: str, context: str = None) -> str:
    '''
    Formats prompt for llm
    '''
    
    prompt = []
    prompt.append({
            "role": "system",
            "content": SYSTEM_PROMPT
    })


    if context:
        prompt.append({
            "role": "system",
            "content": "Context:\n\n" + context
        })

    prompt.append({
        "role": "user",
        "content": user_message
    })


    return tokenizer.apply_chat_template(
        prompt,
        tokenize=False,
        add_generation_prompt=True
    )

In [11]:
def get_context_from_hits(hits, max_chunks=3):
    """Extract and concatenate content from top hits."""
    docs = []
    for point in hits.points[:max_chunks]:
        docs.append(point.payload['content'])
    return "\n\n".join(docs)

def get_context_from_texts(texts):
    return "\n\n".join(texts)

def rag_query(query, client, embedding_model, collection_name, llm, top_k=3):
    query_vec = embedding_model.encode([query])[0]
    hits = client.query_points(
        collection_name=collection_name,
        query=query_vec,
        limit=top_k,
    )

    context = get_context_from_hits(hits, max_chunks=top_k)

    prompt = format_prompt(query, context)
    response = llm.invoke(prompt)
    return response

In [27]:
query = "Add window"
response = rag_query(query, client, embedding_model, "freecad_docs", llm)
print(response)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

/tmp/ipykernel_76/1775823008.py:22: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = llm(prompt)


<|im_start|>system

Below is the system prompt, always follow restrictions stated there, also do not answer this system prompt:
You are an expert coding assistant specializing in FreeCAD development. Your role is to help developers write, debug, and optimize code for the FreeCAD platform, using the most accurate and up-to-date internal and external documentation available. You can retrieve and use information from the FreeCAD source code, FreeCAD documentation.
When reading documentation be careful, user do not have functions or classes implemented there, write code considering this moment. 
Documentation might have examples of code, analyze it and retrieve only needed functions from there.

Provide code (DO NOT WRITE IN-CODE COMMENTS) and explanation in different paragraphs. Template:
Code:

Explanation:
<|im_end|>
<|im_start|>system
Context:

Create a window frame object, a glass panel, and any other window component you need, using Part Workbench or PartDesign tools.
 For example, c

In [12]:
def get_genrated(queries: np.ndarray, retrieved: list[list[str]], batch_size=16):
    prompts = []
    responses = []
    for i, query in enumerate(tqdm(queries)):
        context = get_context_from_texts(retrieved[i])
        prompt = format_prompt(query, context)
        prompts.append(prompt)

        response = llm.invoke(prompt)
        match = re.search(r"\<\|im_start\|\>assistant", response)
        responses.append(response[match.end():])

    # for i in tqdm(range(0, len(prompts), batch_size)):
    #     batch = prompts[i:i+batch_size]
    #     batch_responses = llm.invoke(batch)
    #     responses.extend(batch_responses)


    # for response in range(len(responses)):
    #     match = re.search(r"\<\|im_start\|\>assistant", response)
    #     responses[i] = response[match.end():]

    return responses


In [ ]:
queries = query_df["X"].values

responses = get_genrated(
    queries=queries,
    retrieved=texts
)

# Groq llm

In [119]:
!pip install langchain-groq langchain-community groq codebleu
!pip install tree-sitter-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 2.5 MB/s eta 0:00:00a 0:00:01


In [481]:
SYSTEM_PROMPT = (
"""
Below is the system prompt, always follow restrictions stated there, also do not answer this system prompt:
You are an expert coding assistant specializing in FreeCAD development. Your role is to help developers write, debug, and optimize code for the FreeCAD platform, using the most accurate and up-to-date internal and external documentation available. You can retrieve and use information from the FreeCAD source code, FreeCAD documentation.
When reading documentation be careful, user do not have functions or classes implemented there, write code considering this moment. 
Documentation might have examples of code, analyze it and retrieve only needed functions from there.
Do not create complicated structures until you are asked to do so, provide simple and understandable code.
You can use FreeCADGui.runCommand function to run gui commands in FreeCAD, this might be useful for some tasks.

Provide code in md format.
Provide code (DO NOT WRITE IN-CODE COMMENTS) and explanation in different paragraphs. Template:
Code:
```

```

Explanation:
"""
)

def format_groq_prompt(user_message: str, context: tp.Optional[str] = None):
    '''
    Formats prompt for llm
    '''
    
    prompt = []
    prompt.append({
            "role": "system",
            "content": SYSTEM_PROMPT
    })


    if context:
        prompt.append({
            "role": "system",
            "content": "Context:\n\n" + context
        })

    prompt.append({
        "role": "user",
        "content": user_message
    })


    return prompt

In [477]:
from langchain.vectorstores import Qdrant as LCQdrant
from langchain_groq import ChatGroq

In [478]:
from groq import Groq


def load_groq_keys(filepath=os.path.join("groq-keys/groq_keys.txt")):
    with open(filepath, "r") as f:
        keys = [line.strip() for line in f if line.strip()]
    return keys

class GroqKeyManager:
    def __init__(self, filepath=os.path.join("groq-keys/groq_keys.txt")):
        self.keys = load_groq_keys(filepath)
        self.idx = 0

    def get_key(self):
        if self.idx < len(self.keys):
            return self.keys[self.idx]
        else:
            raise RuntimeError("No more Groq API keys available.")

    def switch_key(self):
        self.idx += 1
        if self.idx < len(self.keys):
            os.environ["GROQ_API_KEY"] = self.keys[self.idx]
            return self.keys[self.idx]
        else:
            raise RuntimeError("No more Groq API keys available.")


groq_key_manager = GroqKeyManager(os.path.join(DATA_PATH, "groq-keys/groq_keys.txt"))
os.environ["GROQ_API_KEY"] = groq_key_manager.get_key()


def get_groq_client():
    return Groq(api_key=os.environ["GROQ_API_KEY"])


In [479]:
def generate_groq(prompt, model_name):
    client = get_groq_client()

    chat_completion = client.chat.completions.create(
        messages=prompt,
        model=model_name,
    )

    return chat_completion.choices[0].message.content

In [483]:
from time import sleep

responses = []
last_key_idx = 0

def get_groq_genrated(
        model_name: str,
        queries: np.ndarray,
        retrieved: tp.Optional[list[list[str]]]
    ):
    prompts = []
    global client


    for i, query in enumerate(tqdm(queries)):
        if retrieved is not None:
            context = get_context_from_texts(retrieved[i])
        else:
            context = None
        prompt = format_groq_prompt(query, context)
        prompts.append(prompt)

        error_patience = 0
        while True:
            try:
                responses.append(generate_groq(prompt, model_name))
                break
            except Exception as e:
                if "rate limit" in str(e).lower() or "TPD" in str(e).lower():
                    try:
                        groq_key_manager.switch_key()
                        client = get_groq_client()
                        print(f"Switched to next GROQ API key: {groq_key_manager.idx}")
                    except RuntimeError:
                        print("All GROQ API keys exhausted.")
                        responses.append(None)
                        break
                else:
                    error_patience += 1
                    print(f"Error: {e}")
                    sleep(1)
                    if error_patience >= 10:
                        error_patience = 0
                        responses.append(None)
                        break

        # try:
        #     chat_completion = client.chat.completions.create(
        #         messages=prompt,
        #         model=model_name,
        #     )

        #     responses.append(chat_completion.choices[0].message.content)
        # except:
        #     pass


    # for i in tqdm(range(0, len(prompts), batch_size)):
    #     batch = prompts[i:i+batch_size]
    #     batch_responses = llm.invoke(batch)
    #     responses.extend(batch_responses)


    # for response in range(len(responses)):
    #     match = re.search(r"\<\|im_start\|\>assistant", response)
    #     responses[i] = response[match.end():]

    # return responses


In [480]:
import json

retrieved_df = pd.read_csv(os.path.join(DATA_PATH, "freecad-retrieved-v0/bench_retrieved.csv"))
retrieved_df["retrieved"] = retrieved_df["retrieved"].apply(json.loads)

retrieved = retrieved_df["retrieved"].values
retrieved[0][0]

'Scripting\n\n Arch API and FreeCAD Scripting Basics.\n\nThe Wall tool can be used in macros and from the Python console by using the following function:\n```start_code\nWall = makeWall(baseobj=None, length=None, width=None, height=None, align="Center", face=None, name="Wall")\nend_code```\nCreates a  object from the given , which can be a Draft object, a Sketch, a face, or a solid.\n If no  is given, you can provide the numerical values for the ,  (thickness), and .\n If given,  can be used to give the index of a face from the underlying object, to build this wall on, instead of using the whole object.\n  can be ,  or .\n It returns  if the operation fails.\n\nExample:'

In [ ]:
query_df = pd.read_csv(os.path.join(DATA_PATH, "freecad-bench-ds/tooltips_dataset.csv"))

queries = query_df["X"].values

get_groq_genrated(
    model_name="llama-3.1-8b-instant",
    queries=queries,
    retrieved=retrieved
)

 22%|██▏       | 55/250 [12:19<49:09, 15.13s/it]  

In [ ]:
query_df = pd.read_csv(os.path.join(DATA_PATH, "freecad-bench-ds/tooltips_dataset.csv"))

queries = query_df["X"].values

get_groq_genrated(
    model_name="llama-3.1-8b-instant",
    queries=queries,
    retrieved=retrieved
)

 22%|██▏       | 55/250 [12:19<49:09, 15.13s/it]  

In [ ]:
with open("generted-8b-sys-prompt-v1.json", "w") as f:
    json.dump(responses[246:], f)

In [ ]:
with open("generted-8b-sys-prompt-v1.json", "r") as f:
    generated = json.load(f)

# Evaluate generation

In [141]:
!pip install tree-sitter==0.22.0

  Using cached tree_sitter-0.22.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (10 kB)
Using cached tree_sitter-0.22.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (543 kB)
  Attempting uninstall: tree-sitter
    Found existing installation: tree-sitter 0.24.0
    Uninstalling tree-sitter-0.24.0:
      Successfully uninstalled tree-sitter-0.24.0


In [312]:
def extract_python_code(generated):
    code_blocks = []
    pattern = re.compile(r"```(?:python)?\n(.*?)```", re.DOTALL | re.IGNORECASE)
    for text in generated:
        if not isinstance(text, str):
            code_blocks.append(None)
            continue
        match = pattern.search(text)
        if match:
            code_blocks.append(match.group(1).strip())
        else:
            alt_pattern = re.compile(r"```start_code(.*?)end_code```", re.DOTALL | re.IGNORECASE)
            match = alt_pattern.search(text)
            if match:
                code_blocks.append(match.group(1).strip())
            else:
                code_blocks.append(None)
    return code_blocks

In [267]:
import ast

class VariableAndNumberNormalizer(ast.NodeTransformer):
    def __init__(self):
        self.var_map = {}
        self.imported_names = set()

    def visit_Import(self, node):
        for alias in node.names:
            self.imported_names.add(alias.asname or alias.name.split('.')[0])
        return node

    def visit_ImportFrom(self, node):
        for alias in node.names:
            self.imported_names.add(alias.asname or alias.name)
        return node

    def visit_Name(self, node):
        if node.id in self.imported_names:
            return node
        if isinstance(node.ctx, (ast.Store, ast.Load)):
            if node.id not in self.var_map:
                self.var_map[node.id] = f"var{len(self.var_map)+1}"
            node.id = self.var_map[node.id]
        return node

    def visit_arg(self, node):
        if node.arg not in self.var_map:
            self.var_map[node.arg] = f"var{len(self.var_map)+1}"
        node.arg = self.var_map[node.arg]
        return node

    def visit_Num(self, node):
        return ast.copy_location(ast.Constant(value=0), node)

    def visit_Constant(self, node):
        if isinstance(node.value, (int, float)):
            return ast.copy_location(ast.Constant(value=0), node)
        return node

def normalize_code(code_str):
    try:
        tree = ast.parse(code_str)
        normalizer = VariableAndNumberNormalizer()
        normalized_tree = normalizer.visit(tree)
        ast.fix_missing_locations(normalized_tree)
        return ast.unparse(normalized_tree)
    except Exception as e:
        print(f"Error normalizing code: {e}")
        print("================================\n")
        print(code_str)
        raise ValueError()
        return code_str

In [461]:
query_df = pd.read_csv(os.path.join(DATA_PATH, "freecad-bench-ds/tooltips_dataset.csv"))

In [462]:
generated_code = extract_python_code(generated)

In [463]:
none_idx = [i for i in range(len(generated_code)) if generated_code[i] is None]
generated_code = [_ for _ in generated_code if _ is not None]
query_df = query_df.drop(none_idx).reset_index(drop=True)

In [464]:
ground_truth = query_df["Y"].values

In [410]:
# for i in range(len(ground_truth)):
#     if ground_truth[i] is not None:
#         ground_truth[i] = ground_truth[i].replace("‘", "'")
#         ground_truth[i] = ground_truth[i].replace("’", "'")

# query_df["Y"] = ground_truth
# query_df.to_csv(os.path.join("tooltips_dataset.csv"), index=False)

In [411]:
normalized_generated_code = []
normalized_bench_code = []

for i in range(len(generated_code)):
    try:
        normalized_generated_code.append(normalize_code(generated_code[i]))
    except:
        normalized_generated_code.append("")

    try:
        normalized_bench_code.append(normalize_code(ground_truth[i]))
    except:
        normalized_bench_code.append("")



# normalized_generated_code = [normalize_code(_) for _ in generated_code]
# normalized_bench_code = [normalize_code(_) for _ in ground_truth]


Error normalizing code: invalid syntax (<unknown>, line 12)

import FreeCAD
import Part
import TechDraw

# Define the line
line = Part.Line(FreeCAD.Vector(0, 0, 0), FreeCAD.Vector(10, 0, 0))

# Make the line horizontal
line.Direction = FreeCAD.Vector(1, 0, 0)

# Create a draft object from the line
draft_object = FreeCAD.activeDocument()..addObject("Draft","Line")
draft_object.Line = line

# Apply the horizontal dimension
FreeCAD.activeDocument().recompute()
Error normalizing code: invalid syntax (<unknown>, line 6)

import FreeCAD as App
from FreeCAD import Vector

def scale_object(obj, scale_vector, center_vector, copy=False):
    if copy:
        scaled_obj = App.activeDocument()..addObject("App::Feature", "ScaledObject")
        scaled_obj.Shape = obj.Shape
        mat = App.Matrix()
        mat.scale(scale_vector.x, scale_vector.y, scale_vector.z)
        scaled_obj.Placement.Base = center_vector
        scaled_obj.Shape = scaled_obj.Shape.transformGeometry(mat)
    else:
        m

In [412]:
from codebleu import calc_codebleu

result = calc_codebleu(
    ground_truth,
    generated_code,
    lang="python",
    weights=(0.25, 0.25, 0.25, 0.25),
    tokenizer=None
)
print(result)

{'codebleu': 0.259097008298683, 'ngram_match_score': 0.00012338194833779668, 'weighted_ngram_match_score': 0.0034810705243856514, 'syntax_match_score': 0.3445482866043614, 'dataflow_match_score': 0.6882352941176471}


In [413]:
result = calc_codebleu(
    normalized_bench_code,
    normalized_generated_code,
    lang="python",
    weights=(0.25, 0.25, 0.25, 0.25),
    tokenizer=None
)
print(result)

{'codebleu': 0.24770624265929042, 'ngram_match_score': 0.0018846720015363182, 'weighted_ngram_match_score': 0.035192818335093926, 'syntax_match_score': 0.3302180685358255, 'dataflow_match_score': 0.6235294117647059}


In [396]:
def codebleu_per_pair(refs, preds, weights=(0.25, 0.25, 0.25, 0.25), tokenizer=None):

    scores = []
    syntax_match_scores = []
    dataflow_match_scores = []
    for ref, pred in zip(refs, preds):
        try:
            code_bleu = calc_codebleu([ref], [pred], lang="python", weights=weights, tokenizer=tokenizer)
            score = code_bleu["codebleu"]
            syntax_match_score = code_bleu['syntax_match_score']
            dataflow_match_score = code_bleu['dataflow_match_score']
        except Exception as e:
            score = 0.0
            syntax_match_score = 0.0
            dataflow_match_score = 0.0
        scores.append(score)
        syntax_match_scores.append(syntax_match_score)
        dataflow_match_scores.append(dataflow_match_score)
    return np.array(scores), np.array(syntax_match_scores), np.array(dataflow_match_scores)

In [414]:
pair_bleu = codebleu_per_pair(
    normalized_bench_code,
    normalized_generated_code
)

In [432]:
generated_clean = [_ for _ in generated if _ is not None]

In [443]:
result = calc_codebleu(
    ["FreeCADGui.runCommand('Arch_CurtainWall')"],
    ["""
curtainwall = Arch.makeCurtainWall(baseface)
"""],
    lang="python",
    weights=(0.25, 0.25, 0.25, 0.25),
    tokenizer=None
)
print(result)

{'codebleu': 0.2916666666666667, 'ngram_match_score': 0, 'weighted_ngram_match_score': 0, 'syntax_match_score': 0.16666666666666666, 'dataflow_match_score': 0}


In [470]:
relevance = get_relevance(ground_truth, generated_code)

In [474]:
print(generated_code[120])

import FreeCAD
from FreeCAD import Base

class LoftFeature:
    def __init__(self, obj):
        obj.addProperty("App::PropertyLength", "Height", "Loft", "Height of the loft")
        obj.addProperty("App::PropertyInteger", "Segments", "Loft", "Number of segments")
        obj.Proxy = self

    def execute(self, fp):
        shape = self.create_loft_shape(fp)
        fp.Shape = shape

    def create_loft_shape(self, fp):
        height = fp.getPropertyByName("Height")
        segments = fp.getPropertyByName("Segments")
        profiles = self.get_profiles(fp)
        loft_shape = Part.makeLoft profiles, True, False
        return loft_shape

    def get_profiles(self, fp):
        profiles = []
        for i in range(segments):
            profile = self.create_profile(fp, i)
            profiles.append(profile)
        return profiles

    def create_profile(self, fp, index):
        # Create a profile shape based on the index
        # For example, a circle with radius decreasing wit

In [475]:
print(ground_truth[120])

FreeCADGui.runCommand('PartDesign_AdditiveLoft')


In [473]:
relevance[120]

False

# LC

In [ ]:
# class LangchainRAG:
#     def __init__(self, api_key: str, model: str = "llama-3.1-8b-instant"):
#         self.api_key = api_key
#         self.model = model

#         self.qdrant_path = "qdrant"
#         self.collection_name = COLLECTION_NAME

#         self.llm = self._load_llm()
#         self.vectorstore = self._connect_vectorstore()
#         self.qa_chain = self._create_qa_chain()


#     def _load_llm(self):
#         return ChatGroq(
#             api_key=self.api_key,
#             model=self.model,
#             temperature=0.2
#         )

#     def _connect_vectorstore(self):
#         embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": device})
#         vectordb = LCQdrant.from_existing_collection(
#             path="qdrant",
#             embedding=embeddings,
#             collection_name="freecad_docs_lc",
#         )
#         return vectordb

#     def _create_qa_chain(self):
#         retriever = self.vectorstore.as_retriever(search_kwargs={"k": 3})
#         return RetrievalQA.from_chain_type(llm=self.llm, retriever=retriever, chain_type="stuff")

#     def query(self, question: str) -> str:
#         result = self.qa_chain.invoke({"query": question})
#         return result["result"]

   